## Setup and Connection

In [1]:
import json
import re
from datetime import date, datetime
from pathlib import Path
from pprint import pprint
from pymongo import MongoClient

## Load the data

In [2]:
# Connect to local MongoDB
client = MongoClient('localhost', 27017)

# Create/access database and collection
db = client['project']
collection = db['credit_applications']
print("Connected to MongoDB successfully!")

Connected to MongoDB successfully!


In [3]:
# Load the json file
current_dir = Path.cwd()
repo_root = current_dir.parent
json_path = repo_root / "data" / "raw_credit_applications.json"

with open(json_path, 'r') as file:
    data = json.load(file)

# Prepare the data by renaming the original '_id' 
# This prevents collisions while preserving the reference
for record in data:
    if '_id' in record:
        record['app_id'] = record.pop('_id')

# Clear the Collection
collection.delete_many({})

# Insert the data into a Collection
try:
    collection.insert_many(data)
    print(f"Successfully inserted {len(data)} documents.")
except Exception as e:
    print(f"An error occurred: {e}")

Successfully inserted 502 documents.


## Quick Data Overview

In [4]:
# View a sample document to understand the structure
sample = collection.find_one()
pprint(sample)

{'_id': ObjectId('69a1f163ba6652aaaba7fb90'),
 'app_id': 'app_200',
 'applicant_info': {'date_of_birth': '2001-03-09',
                    'email': 'jerry.smith17@hotmail.com',
                    'full_name': 'Jerry Smith',
                    'gender': 'Male',
                    'ip_address': '192.168.48.155',
                    'ssn': '596-64-4340',
                    'zip_code': '10036'},
 'decision': {'loan_approved': False,
              'rejection_reason': 'algorithm_risk_score'},
 'financials': {'annual_income': 73000,
                'credit_history_months': 23,
                'debt_to_income': 0.2,
                'savings_balance': 31212},
 'processing_timestamp': '2024-01-15T00:00:00Z',
 'spending_behavior': [{'amount': 480, 'category': 'Shopping'},
                       {'amount': 790, 'category': 'Rent'},
                       {'amount': 247, 'category': 'Alcohol'}]}


## Audit Query 1: Find Duplicates

In [5]:
pipeline = [
    {
        "$group": {
            "_id": "$app_id",      # Group by the original ID field
            "count": {"$sum": 1},      # Count how many documents have this ID
            "docs": {"$push": "$_id"}  # Store the new ObjectIds for reference
        }
    },
    {
        "$match": {
            "count": {"$gt": 1}        # Only return those that appear more than once
        }
    }
]

duplicates = list(db.credit_applications.aggregate(pipeline))

print(f"Found {len(duplicates)} duplicate IDs.")
for entry in duplicates:
    print(f"ID: {entry['_id']} | Occurrences: {entry['count']}")

Found 2 duplicate IDs.
ID: app_001 | Occurrences: 2
ID: app_042 | Occurrences: 2


In [6]:
for entry in duplicates:
    app_id = entry['_id']
    # Fetch all versions from the database
    versions = list(db.credit_applications.find({"app_id": app_id}))
    
    print(f"\n{'='*50}")
    print(f"ANALYZING DUPLICATES FOR ID: {app_id}")
    print(f"{'='*50}")
    
    for i, doc in enumerate(versions):
        print(f"\n--- VERSION {i+1} (Internal _id: {doc['_id']}) ---")
        # Calculate 'completeness' (number of fields)
        field_count = len(doc.keys())
        print(f"Field Count: {field_count}")
        pprint(doc)


ANALYZING DUPLICATES FOR ID: app_001

--- VERSION 1 (Internal _id: 69a1f163ba6652aaaba7fd0f) ---
Field Count: 6
{'_id': ObjectId('69a1f163ba6652aaaba7fd0f'),
 'app_id': 'app_001',
 'applicant_info': {'date_of_birth': '1986-05-27',
                    'email': 'stephanie.nguyen47@mail.com',
                    'full_name': 'Stephanie Nguyen',
                    'gender': 'Female',
                    'ip_address': '10.121.120.213',
                    'ssn': '427-90-1892',
                    'zip_code': '90230'},
 'decision': {'loan_approved': False, 'rejection_reason': 'high_dti_ratio'},
 'financials': {'annual_income': 102000,
                'credit_history_months': 37,
                'debt_to_income': 0.42,
                'savings_balance': 0},
 'spending_behavior': [{'amount': 576, 'category': 'Fitness'}]}

--- VERSION 2 (Internal _id: 69a1f163ba6652aaaba7fd57) ---
Field Count: 7
{'_id': ObjectId('69a1f163ba6652aaaba7fd57'),
 'app_id': 'app_001',
 'applicant_info': {'email': '

For Application app_042 Joseph Lopez we should keep Version 2 because it contains the exact same personal and financial data as the first version but adds a Resubmission note which provides better context for the record.

For Application app_001 Stephanie Nguyen we should keep Version 1 because it contains critical personal information like the SSN and date of birth which is missing from the second version despite the error note.

The general rule for this is to prioritize the most complete record to ensure that no vital applicant information is lost.

In [7]:
# Remove the version of app_001 that has the 'DUPLICATE_ENTRY_ERROR' note
db.credit_applications.delete_one({
    "app_id": "app_001", 
    "notes": "DUPLICATE_ENTRY_ERROR"
})

# Remove the version of app_042 that is missing the 'RESUBMISSION' note
db.credit_applications.delete_one({
    "app_id": "app_042", 
    "notes": {"$exists": False}
})

print(f"Updated document count: {db.credit_applications.count_documents({})}")

Updated document count: 500


In [8]:
# Find duplicate SSNs - each person should appear only once!
pipeline_duplicates = [
    {
        "$group": {
            "_id": "$applicant_info.ssn",
            "count": {"$sum": 1},
            "names": {"$push": "$applicant_info.full_name"}
        }
    },
    {
        "$match": {
            "count": {"$gt": 1}
        }
    },
    {
        "$sort": {"count": -1}
    }
]

duplicates = list(collection.aggregate(pipeline_duplicates))

print(f"Found {len(duplicates)} duplicate SSNs:")
for dup in duplicates[:5]:  # Show first 5
    print(f"  SSN: {dup['_id']} - Count: {dup['count']} - Names: {dup['names']}")

Found 3 duplicate SSNs:
  SSN: None - Count: 4 - Names: ['Margaret Williams', 'Carolyn Martin', 'Larry Williams', 'Brandon Moore']
  SSN: 937-72-8731 - Count: 2 - Names: ['Sandra Smith', 'Samuel Hill']
  SSN: 780-24-9300 - Count: 2 - Names: ['Susan Martinez', 'Gary Wilson']


In [9]:
# Identify the application IDs for the records that are incomplete or conflicting
invalid_app_ids = [
    "app_075", "app_120", "app_268", "app_165", # Missing SSNs
    "app_101", "app_234",                       # SSN Conflict for 937-72-8731
    "app_088", "app_016"                        # SSN Conflict for 780-24-9300
]

# Delete these records from the collection
result = db.credit_applications.delete_many({"app_id": {"$in": invalid_app_ids}})

print(f"Cleanup finished. Removed {result.deleted_count} invalid records.")
print(f"Final document count: {db.credit_applications.count_documents({})}")

Cleanup finished. Removed 8 invalid records.
Final document count: 492


## Audit Query 2: Check Consistency

**Data Quality Dimension:** Consistency  
**Issue:** Same field having different encodings (e.g., "Male" vs "M")

In [10]:
# How many different gender values exist?
pipeline_gender_consistency = [
    {
        "$group": {
            "_id": "$applicant_info.gender",
            "count": {"$sum": 1}
        }
    },
    {
        "$sort": {"count": -1}
    }
]

gender_values = list(collection.aggregate(pipeline_gender_consistency))

print("Gender value distribution:")
print("Expected: 2 distinct values (Male, Female)")
print(f"Actual: {len(gender_values)} distinct values")
print()
for gv in gender_values:
    print(f"  '{gv['_id']}': {gv['count']} records")

Gender value distribution:
Expected: 2 distinct values (Male, Female)
Actual: 4 distinct values

  'Male': 192 records
  'Female': 191 records
  'F': 57 records
  'M': 52 records


In [11]:
# Standardize 'F' to 'Female'
collection.update_many(
    {"applicant_info.gender": "F"},
    {"$set": {"applicant_info.gender": "Female"}}
)

# Standardize 'M' to 'Male'
collection.update_many(
    {"applicant_info.gender": "M"},
    {"$set": {"applicant_info.gender": "Male"}}
)

# Handle missing values (e.g., set to 'Unknown' or drop)
collection.update_many(
    {"applicant_info.gender": ""},
    {"$set": {"applicant_info.gender": "Unknown"}}
)

UpdateResult({'n': 0, 'nModified': 0, 'ok': 1.0, 'updatedExisting': False}, acknowledged=True)

In [12]:
# How many different gender values exist?
pipeline_gender_consistency = [
    {
        "$group": {
            "_id": "$applicant_info.gender",
            "count": {"$sum": 1}
        }
    },
    {
        "$sort": {"count": -1}
    }
]

gender_values = list(collection.aggregate(pipeline_gender_consistency))

print("Gender value distribution:")
print("Expected: 2 distinct values (Male, Female)")
print(f"Actual: {len(gender_values)} distinct values")
print()
for gv in gender_values:
    print(f"  '{gv['_id']}': {gv['count']} records")

Gender value distribution:
Expected: 2 distinct values (Male, Female)
Actual: 2 distinct values

  'Female': 248 records
  'Male': 244 records


In [13]:
# Detect records with non-standard DOB formats
REGEX_PATTERN = r"^\d{4}-\d{2}-\d{2}$"

KNOWN_FORMATS = [
    "%d/%m/%Y",   # e.g. 15/03/1990
    "%m-%d-%Y",   # e.g. 03-15-1990
    "%d-%m-%Y",   # e.g. 15-03-1990
    "%B %d, %Y",  # e.g. March 15, 1990
    "%d %B %Y",   # e.g. 15 March 1990
    "%Y/%m/%d",   # e.g. 1990/03/15
]

inconsistent_records = list(
    collection.find(
        {"applicant_info.date_of_birth": {"$not": {"$regex": REGEX_PATTERN}}}
    )
)

print(f"Found {len(inconsistent_records)} records with non-standard DOB formats.")
for doc in inconsistent_records:
    app_id = doc.get("_id", "unknown")
    dob = doc.get("applicant_info", {}).get("date_of_birth", "missing")
    print(f"  ID: {app_id} | DOB: {dob}")# Find any DOB that does NOT follow the YYYY-MM-DD format
regex_pattern = "^\\d{4}-\\d{2}-\\d{2}$"

inconsistent_formats = list(collection.find({
    "applicant_info.date_of_birth": {"$not": {"$regex": regex_pattern}}
}))

print(f"Found {len(inconsistent_formats)} records with non-standard formats.")
for doc in inconsistent_formats:
    print(f"ID: {doc['app_id']} | DOB: {doc['applicant_info']['date_of_birth']}")

Found 157 records with non-standard DOB formats.
  ID: 69a1f163ba6652aaaba7fb95 | DOB: 14/02/1982
  ID: 69a1f163ba6652aaaba7fb96 | DOB: 28/01/1990
  ID: 69a1f163ba6652aaaba7fb9b | DOB: 01/12/1978
  ID: 69a1f163ba6652aaaba7fb9e | DOB: 1990/07/26
  ID: 69a1f163ba6652aaaba7fba5 | DOB: 18/07/1979
  ID: 69a1f163ba6652aaaba7fba7 | DOB: 20/04/1979
  ID: 69a1f163ba6652aaaba7fbb0 | DOB: 1986/11/20
  ID: 69a1f163ba6652aaaba7fbb3 | DOB: 1995/05/07
  ID: 69a1f163ba6652aaaba7fbb4 | DOB: 03/20/1968
  ID: 69a1f163ba6652aaaba7fbb5 | DOB: 20/07/1997
  ID: 69a1f163ba6652aaaba7fbb7 | DOB: 1987/06/28
  ID: 69a1f163ba6652aaaba7fbb9 | DOB: 1988/04/06
  ID: 69a1f163ba6652aaaba7fbba | DOB: 03/10/1981
  ID: 69a1f163ba6652aaaba7fbbc | DOB: 14/06/1975
  ID: 69a1f163ba6652aaaba7fbbd | DOB: 04/20/1994
  ID: 69a1f163ba6652aaaba7fbc1 | DOB: 04/08/1979
  ID: 69a1f163ba6652aaaba7fbcb | DOB: 29/12/1982
  ID: 69a1f163ba6652aaaba7fbcc | DOB: 14/12/1987
  ID: 69a1f163ba6652aaaba7fbd5 | DOB: 30/09/1978
  ID: 69a1f163ba6652

In [14]:
# Normalise DOB and compute age for all records 
count_normalised = 0
count_age_added = 0
count_skipped = 0

all_records = list(collection.find({}))

for doc in all_records:
    app_id = doc.get("_id")
    raw_dob = doc.get("applicant_info", {}).get("date_of_birth")

    already_standard = bool(re.match(REGEX_PATTERN, raw_dob or ""))

    # Parse the date
    parsed_date = None

    if already_standard:
        try:
            parsed_date = datetime.strptime(raw_dob, "%Y-%m-%d")
        except (ValueError, TypeError):
            pass
    else:
        for fmt in KNOWN_FORMATS:
            try:
                parsed_date = datetime.strptime(raw_dob, fmt)
                break  # Stop at the first successful parse
            except (ValueError, TypeError):
                continue

    # Skip if the date could not be parsed
    if parsed_date is None:
        print(f"  [SKIP] ID: {app_id} | Cannot parse DOB: '{raw_dob}'")
        count_skipped += 1
        continue

    # Compute age
    today = date.today()
    age = today.year - parsed_date.year
    # Subtract 1 if the birthday has not yet occurred this calendar year
    if (today.month, today.day) < (parsed_date.month, parsed_date.day):
        age -= 1

    # Build update payload and write to MongoDB
    normalised_dob = parsed_date.strftime("%Y-%m-%d")
    update_fields = {"applicant_info.age": age}

    if not already_standard:
        update_fields["applicant_info.date_of_birth"] = normalised_dob
        count_normalised += 1
        print(
            f"  [FIX] ID: {app_id} | "
            f"'{raw_dob}' -> '{normalised_dob}' | Age: {age}"
        )
    else:
        count_age_added += 1

    collection.update_one(
        {"_id": app_id},
        {"$set": update_fields}
    )

print(f"\nSummary:")
print(f"  DOB normalised : {count_normalised}")
print(f"  Age added      : {count_age_added}")
print(f"  Skipped        : {count_skipped}")


  [FIX] ID: 69a1f163ba6652aaaba7fb95 | '14/02/1982' -> '1982-02-14' | Age: 44
  [FIX] ID: 69a1f163ba6652aaaba7fb96 | '28/01/1990' -> '1990-01-28' | Age: 36
  [FIX] ID: 69a1f163ba6652aaaba7fb9b | '01/12/1978' -> '1978-12-01' | Age: 47
  [FIX] ID: 69a1f163ba6652aaaba7fb9e | '1990/07/26' -> '1990-07-26' | Age: 35
  [FIX] ID: 69a1f163ba6652aaaba7fba5 | '18/07/1979' -> '1979-07-18' | Age: 46
  [FIX] ID: 69a1f163ba6652aaaba7fba7 | '20/04/1979' -> '1979-04-20' | Age: 46
  [FIX] ID: 69a1f163ba6652aaaba7fbb0 | '1986/11/20' -> '1986-11-20' | Age: 39
  [FIX] ID: 69a1f163ba6652aaaba7fbb3 | '1995/05/07' -> '1995-05-07' | Age: 30
  [SKIP] ID: 69a1f163ba6652aaaba7fbb4 | Cannot parse DOB: '03/20/1968'
  [FIX] ID: 69a1f163ba6652aaaba7fbb5 | '20/07/1997' -> '1997-07-20' | Age: 28
  [FIX] ID: 69a1f163ba6652aaaba7fbb7 | '1987/06/28' -> '1987-06-28' | Age: 38
  [FIX] ID: 69a1f163ba6652aaaba7fbb9 | '1988/04/06' -> '1988-04-06' | Age: 37
  [FIX] ID: 69a1f163ba6652aaaba7fbba | '03/10/1981' -> '1981-10-03' | A

In [15]:
# Check for applicants born after 2007 (likely under 18) 
# or before 1920 (likely data entry error)
outlier_pipeline = [
    {
        "$match": {
            "$or": [
                {"applicant_info.date_of_birth": {"$gt": "2007-12-31"}},
                {"applicant_info.date_of_birth": {"$lt": "1920-01-01"}}
            ]
        }
    }
]

outliers = list(collection.aggregate(outlier_pipeline))
print(f"Found {len(outliers)} age outliers.")

for doc in outliers:
    applicant = doc.get("applicant_info", {})

    print(f"--- Outlier Record: {doc.get('_id', 'unknown')} ---")
    print(f"  Name            : {applicant.get('full_name')}",
          f"| DOB: {applicant.get('date_of_birth')}",
          f"| Age: {applicant.get('age')}")
    print(f"  Gender          : {applicant.get('gender')}")
    print()

Found 27 age outliers.
--- Outlier Record: 69a1f163ba6652aaaba7fbb4 ---
  Name            : Angela Carter | DOB: 03/20/1968 | Age: None
  Gender          : Female

--- Outlier Record: 69a1f163ba6652aaaba7fbbd ---
  Name            : Patrick Wilson | DOB: 04/20/1994 | Age: None
  Gender          : Male

--- Outlier Record: 69a1f163ba6652aaaba7fbe8 ---
  Name            : Michelle Perez | DOB: 12/16/1985 | Age: None
  Gender          : Female

--- Outlier Record: 69a1f163ba6652aaaba7fc00 ---
  Name            : Daniel Green | DOB: 02/18/1983 | Age: None
  Gender          : Male

--- Outlier Record: 69a1f163ba6652aaaba7fc03 ---
  Name            : Raymond Hall | DOB: 12/28/1972 | Age: None
  Gender          : Male

--- Outlier Record: 69a1f163ba6652aaaba7fc16 ---
  Name            : Katherine Lopez | DOB: 07/28/1988 | Age: None
  Gender          : Female

--- Outlier Record: 69a1f163ba6652aaaba7fc47 ---
  Name            : Jason Hernandez | DOB: 06/25/1994 | Age: None
  Gender          : 

In [16]:
# Check if any 'annual_income' values are stored as strings instead of numbers
string_income = list(collection.find({"financials.annual_income": {"$type": "string"}}))

print(f"Found {len(string_income)} records where income is a string.")

Found 7 records where income is a string.


In [17]:
# Correct the records by converting the string to a number
for doc in string_income:
    app_id = doc.get("_id")
    raw_income = doc.get("financials", {}).get("annual_income")

    try:
        corrected_income = int(float(raw_income))
        collection.update_one(
            {"_id": app_id},
            {"$set": {"financials.annual_income": corrected_income}}
        )
        print(f"  [FIXED] ID: {app_id} | '{raw_income}' -> {corrected_income}")
    except (ValueError, TypeError):
        print(f"  [SKIP]  ID: {app_id} | Cannot convert '{raw_income}' to integer")

  [FIXED] ID: 69a1f163ba6652aaaba7fc22 | '65000' -> 65000
  [FIXED] ID: 69a1f163ba6652aaaba7fc3b | '73000' -> 73000
  [FIXED] ID: 69a1f163ba6652aaaba7fcc2 | '51000' -> 51000
  [FIXED] ID: 69a1f163ba6652aaaba7fcde | '72000' -> 72000
  [FIXED] ID: 69a1f163ba6652aaaba7fd3f | '80000' -> 80000
  [FIXED] ID: 69a1f163ba6652aaaba7fd4f | '111000' -> 111000
  [FIXED] ID: 69a1f163ba6652aaaba7fd7a | '93000' -> 93000


In [18]:
# Check if any 'annual_income' values are stored as strings instead of numbers
string_income = list(collection.find({"credit_history_months": {"$type": "string"}}))

print(f"Found {len(string_income)} records credit history is a string.")

Found 0 records credit history is a string.


## Save preprocessed data

In [20]:
# Build the output path the same way you built the input path
current_dir = Path.cwd()
repo_root = current_dir.parent
output_path = repo_root / "data" / "cleaned_credit_applications.json"

# Fetch all cleaned records from MongoDB, excluding the internal _id
cleaned_records = list(collection.find({}, {"_id": 0}))

# Save to JSON file
with open(output_path, "w") as f:
    json.dump(cleaned_records, f, indent=4)

print(f"Exported {len(cleaned_records)} cleaned records to {output_path}")

Exported 492 cleaned records to /Users/vanessaweiss/dego-project-team13/data/cleaned_credit_applications.json
